Set True if you want to run a sweep

In [ ]:
do_sweep = True
is_causal = True # Set to False for future leakage

Set systempath

In [ ]:
import sys
sys.path.append("../src")

Import everthing needed

In [ ]:
import pandas as pd
import numpy as np
import os
import wandb
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_, spectral_norm, weight_norm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from transforms.feature_engineering import add_all_features, filter_business_hours, entries_per_day_per_site
from transforms.feature_engineering import (
    CONTINUOUS_FEATURE_COLUMNS,
    CATEGORICAL_FEATURE_COLUMNS,
    CYCLIC_FEATURE_COLUMNS,
    TARGET_COLUMN
)
from evaluation.comp_metrics import evaluate_all_metrics

Set wandb key (don't push to repository)

In [ ]:
os.environ["WANDB_API_KEY"] = "3aaf9f796df65417b3f5f8560b43875171b55805"

Set seed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Load the data

In [ ]:
df_train = pd.read_csv("../data/classification/classification-train.csv")
df_test = pd.read_csv("../data/classification/classification-test.csv")

In [ ]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

In [ ]:
df_train = add_all_features(df_train)

In [ ]:
df_train = filter_business_hours(df_train)

In [ ]:
df_train.columns

In [ ]:
ENTRIES_PER_DAY = entries_per_day_per_site(df_train)

In [ ]:
print(ENTRIES_PER_DAY)

In [ ]:
def prepare_data():
    """Prepare and return all data splits"""
    df_train_site_a = df_train[0:19345]
    df_train_site_b = df_train[19345:38690]
    df_train_site_c = df_train[38690:58035]

    X_continuous_site_a = df_train_site_a[CONTINUOUS_FEATURE_COLUMNS].values
    X_continuous_site_b = df_train_site_b[CONTINUOUS_FEATURE_COLUMNS].values
    X_continuous_site_c = df_train_site_c[CONTINUOUS_FEATURE_COLUMNS].values

    X_categorical_site_a = df_train_site_a[CATEGORICAL_FEATURE_COLUMNS].values
    X_categorical_site_b = df_train_site_b[CATEGORICAL_FEATURE_COLUMNS].values
    X_categorical_site_c = df_train_site_c[CATEGORICAL_FEATURE_COLUMNS].values

    X_cyclic_site_a = df_train_site_a[CYCLIC_FEATURE_COLUMNS].values
    X_cyclic_site_b = df_train_site_b[CYCLIC_FEATURE_COLUMNS].values
    X_cyclic_site_c = df_train_site_c[CYCLIC_FEATURE_COLUMNS].values

    y_site_a = df_train_site_a[TARGET_COLUMN].values.reshape(-1, 1)
    y_site_b = df_train_site_b[TARGET_COLUMN].values.reshape(-1, 1)
    y_site_c = df_train_site_c[TARGET_COLUMN].values.reshape(-1, 1)

    y_site_a_unscaled = y_site_a.copy()

    scaler_X_site_a = StandardScaler()
    scaler_X_site_b = StandardScaler()
    scaler_X_site_c = StandardScaler()

    scaler_y_site_a = StandardScaler()
    scaler_y_site_b = StandardScaler()
    scaler_y_site_c = StandardScaler()

    X_scaled_site_a = scaler_X_site_a.fit_transform(X_continuous_site_a)
    X_scaled_site_b = scaler_X_site_b.fit_transform(X_continuous_site_b)
    X_scaled_site_c = scaler_X_site_c.fit_transform(X_continuous_site_c)

    y_site_a = scaler_y_site_a.fit_transform(y_site_a)
    y_site_b = scaler_y_site_b.fit_transform(y_site_b)
    y_site_c = scaler_y_site_c.fit_transform(y_site_c)

    X_site_a = np.concatenate([X_scaled_site_a, X_categorical_site_a, X_cyclic_site_a], axis=1).astype(np.float32)
    X_site_b = np.concatenate([X_scaled_site_b, X_categorical_site_b, X_cyclic_site_b], axis=1).astype(np.float32)
    X_site_c = np.concatenate([X_scaled_site_c, X_categorical_site_c, X_cyclic_site_c], axis=1).astype(np.float32)

    y_site_a = y_site_a.astype(np.float32)
    y_site_b = y_site_b.astype(np.float32)
    y_site_c = y_site_c.astype(np.float32)

    # Remove NaN entries
    mask_site_a = np.ones(len(X_site_a), dtype=bool)
    mask_site_b = np.ones(len(X_site_b), dtype=bool)
    mask_site_c = np.ones(len(X_site_c), dtype=bool)

    mask_site_a[0:ENTRIES_PER_DAY] = False
    mask_site_b[0:ENTRIES_PER_DAY] = False
    mask_site_c[0:ENTRIES_PER_DAY] = False

    X_site_a = X_site_a[mask_site_a]
    X_site_b = X_site_b[mask_site_b]
    X_site_c = X_site_c[mask_site_c]

    y_site_a = y_site_a[mask_site_a]
    y_site_b = y_site_b[mask_site_b]
    y_site_c = y_site_c[mask_site_c]

    return {
        'X_site_a': X_site_a, 'X_site_b': X_site_b, 'X_site_c': X_site_c,
        'y_site_a': y_site_a, 'y_site_b': y_site_b, 'y_site_c': y_site_c,
        'scaler_X_site_a': scaler_X_site_a, 'scaler_X_site_b': scaler_X_site_b, 'scaler_X_site_c': scaler_X_site_c,
        'scaler_y_site_a': scaler_y_site_a, 'scaler_y_site_b': scaler_y_site_b, 'scaler_y_site_c': scaler_y_site_c,
        'y_site_a_unscaled': y_site_a_unscaled
    }

In [ ]:
def create_sequences(X, y, seq_length):
    sequences_X = []
    sequences_y = []
    
    for i in range(len(X) - seq_length):
        sequences_X.append(X[i:i+seq_length])
        sequences_y.append(y[i+seq_length - 1])

    print(f"Sequences X: {len(sequences_X)}, Sequences Y: {len(sequences_y)}")
    
    return np.array(sequences_X), np.array(sequences_y)

In [ ]:
class PowerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class TemporalBlockDS(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation,
                 dropout, causal=False):
        super().__init__()

        padding = (kernel_size - 1) * dilation
        if not causal:
            padding //= 2

        self.causal = causal
        self.pad_left = padding
        self.pad_right = 0 if causal else padding

        # depthwise
        self.depthwise = nn.Conv1d(
            in_channels,
            in_channels,
            kernel_size,
            padding=0,
            dilation=dilation,
            groups=in_channels,
            bias=False
        )

        # pointwise
        self.pointwise = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )
        
        self.norm1 = nn.BatchNorm1d(out_channels)

        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)

        # residual
        self.residual = nn.Conv1d(in_channels, out_channels, 1) \
            if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        # padding
        x_pad = F.pad(x, (self.pad_left, self.pad_right))

        y = self.depthwise(x_pad)
        y = self.pointwise(y)
        y = self.norm1(y)
        y = self.act(y)
        y = self.dropout(y)

        return y + self.residual(x)

In [ ]:
class ModernTCN(nn.Module):
    """
    Modern TCN with depthwise separable dilated convs, residuals, RevIN optional,
    and configurable readout. Suitable for long-range predictions.
    Input: (B, seq_len, features)
    Output: (B, out_dim) -- by default out_dim=1
    """
    def __init__(self, input_size, seq_len, out_dim=1, hidden_dim=128, num_layers=6,
                 kernel_size=3, dropout=0.1, causal=True,
                 readout='last', channel_growth=True):
        """
        readout: 'last' (use last timestep features), 'global_avg' (avg over time),
                 or 'flatten' (flatten entire representation -- use with caution)
        channel_growth: if True, doubles channels every 2 layers until hidden_dim
                        (helps increase capacity progressively)
        """
        super().__init__()
        self.input_size = input_size
        self.seq_len = seq_len
        self.out_dim = out_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.kernel_size = kernel_size
        self.dropout = dropout
        self.causal = causal
        self.readout = readout

        # Add this line BEFORE the loop
        self.input_projection = nn.Conv1d(input_size, hidden_dim, kernel_size=1)

        layers = []
        in_ch = hidden_dim  # Now we start with hidden_dim channels (e.g., 128)
        for i in range(num_layers):
            out_ch = hidden_dim  # Just keep it constant - no complex growth logic
            
            dilation = 2 ** i
            block = TemporalBlockDS(in_ch, out_ch,
                                    kernel_size=kernel_size,
                                    dilation=dilation,
                                    dropout=dropout,
                                    causal=causal)
            layers.append(block)
            in_ch = out_ch

        self.tcn = nn.Sequential(*layers)

        # readout / head
        if readout == 'last':
            self.head = nn.Sequential(
                nn.Linear(in_ch, in_ch // 2 if in_ch >= 2 else 1),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(in_ch // 2 if in_ch >= 2 else 1, out_dim)
            )
        elif readout == 'global_avg':
            # we will GlobalAvg over time -> shape (B, C)
            self.head = nn.Sequential(
                nn.Linear(in_ch, in_ch // 2 if in_ch >= 2 else 1),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(in_ch // 2 if in_ch >= 2 else 1, out_dim)
            )
        elif readout == 'flatten':
            # flatten (C * T) -> out_dim (be mindful of params)
            self.head = nn.Sequential(
                nn.Linear(in_ch * seq_len, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim)
            )
        else:
            raise ValueError("readout must be 'last', 'global_avg' or 'flatten'")

    def forward(self, x):
        # x: (B, seq_len, features)
        B, T, C = x.shape
        assert C == self.input_size, f"Expected input feature dim {self.input_size}, got {C}"

        # conv expects (B, C, T)
        x = x.transpose(1, 2)

        x = self.input_projection(x)

        # TCN forward
        y = self.tcn(x)   # (B, channels, T)

        # readout
        if self.readout == 'last':
            y = y[:, :, -1]           # (B, channels)
            out = self.head(y)       # (B, out_dim)
        elif self.readout == 'global_avg':
            y = torch.mean(y, dim=2)  # (B, channels)
            out = self.head(y)
        else:  # flatten
            out = y.view(B, -1)
            out = self.head(out)

        # NOTE: We do NOT denormalize predictions automatically, because RevIN
        # was applied to the input features (not the target). If you used RevIN
        # to normalize the target, you should call revin.denorm on a shaped tensor.
        return out

In [ ]:
def create_modern_tcn(input_size, seq_len,
                      out_dim=1,
                      hidden_dim=128,
                      num_layers=6,
                      kernel_size=3,
                      dropout=0.1,
                      causal=is_causal,
                      readout='last'):
    return ModernTCN(input_size=input_size,
                     seq_len=seq_len,
                     out_dim=out_dim,
                     hidden_dim=hidden_dim,
                     num_layers=num_layers,
                     kernel_size=kernel_size,
                     dropout=dropout,
                     causal=causal,
                     readout=readout)


In [ ]:
def get_lr_with_warmup(epoch, base_lr, warmup_epochs):
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return base_lr
    else:
        return base_lr * (epoch + 1) / warmup_epochs

In [ ]:
def train_model(config=None):
    """
    Train the SimpleRNN model using a W&B sweep configuration.

    Includes:
    - Data preparation and scaling
    - Sequence creation
    - Model initialization
    - Training with custom false-positive penalty during non-DR hours
    - Validation, early stopping, and best-model saving
    - W&B metric logging
    """

    # -------------------------------
    # 1. Initialize Weights & Biases
    # -------------------------------
    wandb_run = wandb.init(
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern",
        config=config
    )
    config = wandb.config

    best_model_path = os.path.join(wandb_run.dir, "best_model.pt")

    print("\n" + "=" * 60)
    print("Starting run with config:")
    for k, v in dict(config).items():
        print(f"  {k}: {v}")
    print("=" * 60 + "\n")

    # -------------------------------
    # 2. Load + preprocess data
    # -------------------------------
    data = prepare_data()

    # Create sequences per site
    X_a, y_a = create_sequences(data['X_site_a'], data['y_site_a'], config.sequence_length)
    X_c, y_c = create_sequences(data['X_site_c'], data['y_site_c'], config.sequence_length)
    X_b, y_b = create_sequences(data['X_site_b'], data['y_site_b'], config.sequence_length)

    # Combine A + C for training
    X_train = np.vstack((X_a, X_c))
    y_train = np.vstack((y_a, y_c))

    X_val = X_b
    y_val = y_b

    # -------------------------------
    # 3. Build PyTorch Datasets
    # -------------------------------
    train_dataset = PowerDataset(X_train, y_train)
    val_dataset = PowerDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    # -------------------------------
    # 4. Extract metadata for metrics
    # -------------------------------
    # Inverse-transform building power for NMAE/NRMSE calculations
    train_bp_scaled = X_train[:, -1, 2]  # column index 2 = building power
    val_bp_scaled = X_val[:, -1, 2]

    # Helper array to inverse-transform only building power
    t_train = np.zeros((len(train_bp_scaled), 19))
    t_train[:, 2] = train_bp_scaled
    t_a = t_train[:len(X_a)]
    t_c = t_train[len(X_a):]

    bp_a = data['scaler_X_site_a'].inverse_transform(t_a)[:, 2]
    bp_c = data['scaler_X_site_c'].inverse_transform(t_c)[:, 2]
    train_building_power = np.concatenate([bp_a, bp_c])

    t_val = np.zeros((len(val_bp_scaled), 19))
    t_val[:, 2] = val_bp_scaled
    val_building_power = data['scaler_X_site_b'].inverse_transform(t_val)[:, 2]

    # Demand flag extraction
    train_demand_flags = np.argmax(X_train[:, -1, 30:33], axis=1) - 1
    val_demand_flags = np.argmax(X_val[:, -1, 30:33], axis=1) - 1

    # Site labels for metrics
    train_sites = np.concatenate([
        np.array(['Site A'] * len(bp_a)),
        np.array(['Site C'] * len(bp_c))
    ])
    val_sites = np.array(['Site B'] * len(X_val))

    # -------------------------------
    # 5. Initialize model + optimizer
    # -------------------------------
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = create_modern_tcn(
        input_size=X_train.shape[2],
        seq_len=config.sequence_length,
        out_dim=1,                          # or config.output_size
        hidden_dim=config.out_channels,
        num_layers=config.num_blocks,
        kernel_size=config.kernel_size,
        dropout=config.dropout,
        causal=is_causal,                        # True = causal TCN (like your old one)
        readout='last'                      # try 'global_avg' for long-range
    ).to(device)



    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    # -------------------------------
    # 6. Early stopping setup
    # -------------------------------
    early_stopping_patience = config.patience
    best_nmae = float('inf')
    epochs_without_improvement = 0

    print("Starting training...")

    # -------------------------------
    # 7. Training loop
    # -------------------------------
    for epoch in range(config.num_epochs):

        # Learning rate warmup
        current_lr = get_lr_with_warmup(epoch, config.learning_rate, config.warmup_epochs)
        for pg in optimizer.param_groups:
            pg['lr'] = current_lr

        model.train()
        train_loss = 0
        preds_all = []
        targets_all = []

        # ---------------------------
        # Training batches
        # ---------------------------
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)

            # --- CUSTOM PENALTY ---
            base_loss = criterion(outputs, y_batch)

            # Identify "no DR" samples from one-hot flags
            dr_flags = X_batch[:, -1, 30:33]
            dr_ids = torch.argmax(dr_flags, dim=1)
            mask_no_dr = (dr_ids == 1)

            if mask_no_dr.any():
                preds_no_dr = outputs[mask_no_dr].view(-1)
                penalty_term = torch.mean(preds_no_dr ** 2)
            else:
                penalty_term = torch.zeros(1, device=device)


            penalty_weight = getattr(config, "fp_penalty_weight", 5.0)
            loss = base_loss + penalty_weight * penalty_term
            # ------------------------------------------------

            # Backprop
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), config.gradient_clip_val)
            optimizer.step()

            train_loss += loss.item()
            preds_all.append(outputs.detach().cpu().numpy())
            targets_all.append(y_batch.cpu().numpy())

        train_loss /= len(train_loader)

        # -------------------------------
        # 8. Inverse transform predictions
        # -------------------------------
        preds_all = np.concatenate(preds_all)
        targets_all = np.concatenate(targets_all)

        
        preds_a = data['scaler_y_site_a'].inverse_transform(preds_all[:len(y_a)])
        preds_c = data['scaler_y_site_c'].inverse_transform(preds_all[len(y_a):])
        targs_a = data['scaler_y_site_a'].inverse_transform(targets_all[:len(y_a)])
        targs_c = data['scaler_y_site_c'].inverse_transform(targets_all[len(y_a):])

        train_preds = np.concatenate([preds_a, preds_c])
        train_targets = np.concatenate([targs_a, targs_c])

        # Training metrics
        train_metrics = evaluate_all_metrics(
            y_true=train_targets,
            y_pred=train_preds,
            site_labels=train_sites,
            building_power=train_building_power,
            demand_flags=train_demand_flags
        )

        # -------------------------------
        # 9. Validation
        # -------------------------------
        model.eval()
        val_loss = 0
        val_preds = []
        val_targs = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)

                loss = criterion(outputs, y_batch)
                val_loss += loss.item()

                val_preds.append(outputs.cpu().numpy())
                val_targs.append(y_batch.cpu().numpy())

        val_loss /= len(val_loader)

        val_preds = data['scaler_y_site_b'].inverse_transform(np.concatenate(val_preds))
        val_targs = data['scaler_y_site_b'].inverse_transform(np.concatenate(val_targs))

        val_metrics = evaluate_all_metrics(
            y_true=val_targs.flatten(),
            y_pred=val_preds.flatten(),
            site_labels=val_sites,
            building_power=val_building_power,
            demand_flags=val_demand_flags
        )

        # -------------------------------
        # 10. Log to W&B
        # -------------------------------
        wandb.log({
            "epoch": epoch,
            "learning_rate": current_lr,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/nmae_mean": train_metrics['nmae_mean'],
            "val/nmae_mean": val_metrics['nmae_mean'],
            "train/base_loss": float(base_loss.item()),
            "train/penalty": float(penalty_term.item()),
            "train/total_loss": float(loss.item())
        })

        # Print occasionally
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"\nEpoch {epoch+1}/{config.num_epochs}")
            print(f"LR: {current_lr:.6f}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Val NMAE(mean): {val_metrics['nmae_mean']:.2f}%")

        # -------------------------------
        # 11. Early stopping + save best
        # -------------------------------
        current_val_nmae = val_metrics["nmae_mean"]

        if current_val_nmae < best_nmae:
            best_nmae = current_val_nmae
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved best model at epoch {epoch+1} (Val NMAE: {best_nmae:.2f}%)")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stopping_patience:
            print(f"\nEarly stopping at epoch {epoch+1}. Best Val NMAE: {best_nmae:.2f}%")
            break

    print(f"\nTraining completed! Best Val NMAE: {best_nmae:.2f}%")
    return best_nmae


In [ ]:
if do_sweep:
    sweep_config = {
        "method": "bayes",
        "metric": {"name": "val/nmae_mean", "goal": "minimize"},
        "parameters": {

            # Learning rate (ModernTCN tolerates higher lr)
            "learning_rate": {
                "distribution": "log_uniform_values",
                "min": 4e-5,
                "max": 1e-3
            },

            # Batch size
            "batch_size": {"values": [64]},

            # ModernTCN width (hidden_dim)
            "out_channels": {"values": [32, 48, 64, 96]},

            # Number of TCN layers
            "num_blocks": {"values": [2, 3, 4, 6]},

            # Kernel size — larger receptive field strongly helps ModernTCN
            "kernel_size": {"values": [3, 5, 7]},

            # Dropout — sweet spot is often 0.1–0.3
            "dropout": {
                "distribution": "uniform",
                "min": 0.1,
                "max": 0.3
            },

            # Weight decay — ModernTCN works well with 1e-5 to 1e-3
            "weight_decay": {
                "distribution": "log_uniform_values",
                "min": 1e-6,
                "max": 1e-4
            },

            # Sequence length — long-range performance boost
            "sequence_length": {"values": [36, 48, 72, 96, 144]},

            # Epochs
            "num_epochs": {"value": 200},
            "warmup_epochs": {"value": 10},

            # Gradient clipping
            "gradient_clip_val": {
                "distribution": "uniform",
                "min": 0.3,
                "max": 1.0
            },

            # FP penalty stays the same
            "fp_penalty_weight": {"values": [0.0, 1.0, 5.0, 10.0, 50.0]},

            # Set Patience
            "patience": {"value": 20}
        }
    }

    sweep_id = wandb.sweep(
        sweep_config,
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern"
    )
    print(f"Sweep created: {sweep_id}")

    wandb.agent(
        sweep_id,
        function=train_model,
        count=50
    )

else:
    default_config = {
        "out_channels": 64,       # Wider ModernTCN sweet spot
        "num_blocks": 4,           # Deep but stable with residual gating
        "kernel_size": 5,          # Very strong for long-range dependencies
        "dropout": 0.2,
        "sequence_length": 48,     # More context -> better forecasting
        "batch_size": 64,
        "learning_rate": 1e-4,     # Faster training with soft-stable updates
        "weight_decay": 5e-5,
        "gradient_clip_val": 1.0,
        "num_epochs": 100,
        "warmup_epochs": 10,
        "fp_penalty_weight": 5,
        "patience": 20
    }

    train_model(default_config)